# 04 — Evaluate the Model

**Pipeline:** Dataset → EDA → Preprocessing → Augmentation → Training → **Evaluation → Error Analysis** → Inference → Deployment

Val accuracy chose the checkpoint; the **test split** — untouched until now — is the honest number.
One rule: look at test results, *don't tune on them*. The moment you tweak the config to chase
test accuracy, it stops measuring generalization.

In [ ]:
# run from notebooks/ -- make the repo root importable and the cwd
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

from src.utils import load_config, seed_everything
cfg = load_config("configs/config.yaml")
seed_everything(cfg["seed"])
print("project:", cfg["project_name"], "| data root:", cfg["data"]["root"])

## Metrics + reports

`src.evaluate` writes everything to `outputs/reports/`: per-class precision/recall/F1, the confusion matrix, a top-losses grid, and a CSV of every prediction.

In [ ]:
from src.evaluate import evaluate

result = evaluate(cfg)   # prints test accuracy + per-class report

In [ ]:
from IPython.display import Image as IPyImage, display
from pathlib import Path

reports = Path(cfg["output"]["dir"]) / "reports"
display(IPyImage(reports / "confusion_matrix.png"))

## Error analysis — the step most people skip

The confusion matrix says *which* classes get confused; the top-losses grid shows *the actual images*. Three patterns to look for:

1. **Wrong label** on the image → dataset problem, fix the data
2. **Genuinely ambiguous** (even you can't tell) → the ceiling is the labels, not the model
3. **Model missing something obvious** → more data or targeted augmentation for that case

In [ ]:
display(IPyImage(reports / "top_losses.png"))

## Where does confidence break down?

Correct predictions should be confident, wrong ones uncertain. If the model is *confidently wrong*, calibration is off — and a deployment confidence threshold would help.

In [ ]:
import csv
import matplotlib.pyplot as plt

with open(reports / "predictions.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
right = [float(r["confidence"]) for r in rows if r["true"] == r["pred"]]
wrong = [float(r["confidence"]) for r in rows if r["true"] != r["pred"]]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(right, bins=20, alpha=0.7, label=f"correct ({len(right)})")
ax.hist(wrong, bins=20, alpha=0.7, label=f"wrong ({len(wrong)})")
ax.set_xlabel("prediction confidence"); ax.set_ylabel("count"); ax.legend()
ax.set_title("confidence: correct vs wrong predictions")
plt.tight_layout(); plt.show()

## Before shipping

- [ ] Test accuracy is close to val accuracy (a big gap means the split leaked or val was tuned to death)
- [ ] No single class is carrying the failure (check the per-class report)
- [ ] You looked at the top-losses grid and know *why* the worst errors happen

Next: **[05 — Inference](05_inference.ipynb)** — using the model on new images, and the demo app.